# 📈 Moving Average Crossover — Golden Cross / Death Cross

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VesperaSystems/vespera-strategies/blob/main/moving-average-crossover/strategy.ipynb)

Part of the [Vespera Systems](https://vesperasystems.com) quant lab — DIY, math-forward, build-in-public.

This notebook walks through a classic trend-following strategy from first principles:

1. **Theory** — what a moving-average crossover is and why it might work
2. **Signals** — compute them on real market data
3. **Backtest** — simulate the strategy and compare it to buy-and-hold
4. **Metrics** — Sharpe ratio, CAGR, max drawdown
5. **Honesty** — where this strategy breaks


## 🧮 Theory

A **Simple Moving Average (SMA)** smooths price over a window of $n$ days:

$$\text{SMA}_n(t) = \frac{1}{n} \sum_{i=0}^{n-1} P(t - i)$$

where $P(t)$ is the closing price at time $t$.

We track two of them — a fast one (50-day) and a slow one (200-day):

- 🟢 **Golden Cross** — the short SMA crosses **above** the long SMA → uptrend beginning → *buy signal*
- 🔴 **Death Cross** — the short SMA crosses **below** the long SMA → downtrend beginning → *sell signal*

The intuition: when recent prices are persistently above the long-run average, momentum is on your side. The cost: moving averages *lag* — you always enter after the trend has already started, and whipsaw markets eat you alive.


In [ ]:
# Setup — in Colab this installs the Vespera strategy library from GitHub.
# Locally (repo cloned + `pip install -e .`) it just imports.
try:
    import vespera_strategies
except ImportError:
    %pip install -q git+https://github.com/VesperaSystems/vespera-strategies.git
    import vespera_strategies

from vespera_strategies import (
    fetch_stock_data,
    apply_sma_crossover,
    compute_backtest,
    detect_crossovers,
    latest_signal,
    summarize,
)
import pandas as pd
import matplotlib.pyplot as plt

print("vespera_strategies ready 🤘")

In [ ]:
# Parameters — change these and re-run everything below
TICKER = "SPY"
START = "2015-01-01"
END = None  # None = up to today
SHORT_WINDOW = 50
LONG_WINDOW = 200

In [ ]:
# 1. Fetch data and compute signals
df = fetch_stock_data(TICKER, START, END)
df = apply_sma_crossover(df, short_window=SHORT_WINDOW, long_window=LONG_WINDOW)
df[["Close", "SMA_Short", "SMA_Long", "Signal"]].tail()

In [ ]:
# 2. Price, both SMAs, and every crossover event
events = detect_crossovers(df)

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(df.index, df["Close"], label="Close", lw=1, alpha=0.7)
ax.plot(df.index, df["SMA_Short"], label=f"SMA {SHORT_WINDOW}", lw=1.2)
ax.plot(df.index, df["SMA_Long"], label=f"SMA {LONG_WINDOW}", lw=1.2)

if not events.empty:
    golden = events[events["event"] == "golden_cross"]
    death = events[events["event"] == "death_cross"]
    ax.scatter(golden.index, golden["close"], marker="^", s=140, color="green", zorder=5, label="Golden Cross")
    ax.scatter(death.index, death["close"], marker="v", s=140, color="red", zorder=5, label="Death Cross")

ax.set_title(f"{TICKER} — SMA {SHORT_WINDOW}/{LONG_WINDOW} crossovers")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

events

## 📊 Backtest assumptions

| Assumption | Description |
|---|---|
| ✅ One position at a time | Long only (shorting comes later) |
| ✅ All-in/all-out trades | No position sizing or scaling |
| ✅ Next-bar execution | We act the day *after* a signal (no lookahead) |
| ❌ No slippage or fees | Unrealistic — real results would be worse |
| ❌ No volume confirmation | Price-only signal |

Keep these in mind: every simplification flatters the strategy.


In [ ]:
# 3. Backtest — strategy equity curve vs buy-and-hold
df = compute_backtest(df)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df["Cumulative_Market"], label="Buy & Hold", lw=1.4)
ax.plot(df.index, df["Cumulative_Strategy"], label="MA Crossover strategy", lw=1.4)
ax.set_title(f"{TICKER} — growth of $1")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Performance metrics — strategy vs buy-and-hold
stats = summarize(df)
table = pd.DataFrame(stats).rename(
    index={
        "total_return": "Total return",
        "cagr": "CAGR",
        "sharpe": "Sharpe (annualized)",
        "max_drawdown": "Max drawdown",
    },
    columns={"strategy": "Strategy", "buy_hold": "Buy & Hold"},
)
styled = table.copy().astype(object)
for row in styled.index:
    fmt = "{:.2f}" if "Sharpe" in row else "{:.2%}"
    styled.loc[row] = [fmt.format(v) for v in table.loc[row]]
styled

In [ ]:
# 5. Where does the strategy stand right now?
latest_signal(df)

## 📋 What to take away

- Crossovers often **underperform buy-and-hold in strong bull markets** (you're out of the market after every death cross while it recovers).
- Where they help is **avoiding the worst drawdowns** — compare the max drawdown rows above.
- Sideways/choppy markets produce **whipsaws**: repeated crossings that trigger trades with no trend behind them.
- Try different tickers and windows above. Does 20/100 behave differently? Does it work on volatile single names as well as on index ETFs?

**Next in the lab:** RSI + Bollinger divergence, volatility breakout, mean reversion — see the [repo](https://github.com/VesperaSystems/vespera-strategies).


## 🛰 Push this run to Vespera Mission Control (optional)

If you have an API key for the Vespera hub, uncomment below to store this backtest — it will show up on the strategy's lab page and be queryable in chat.


In [ ]:
# from vespera_strategies.ma_crossover import run_ma_crossover
# from vespera_strategies.reporting import post_backtest_run
#
# df, summary = run_ma_crossover(TICKER, START, END, SHORT_WINDOW, LONG_WINDOW)
# post_backtest_run(
#     df, summary,
#     base_url="https://YOUR-MISSION-CONTROL-URL",
#     api_key="YOUR-API-KEY",  # never commit this
#     source="colab",
# )